# B2-024 — Review and Self-Quiz

*60 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

## Formula and protocol sheet

- **Device pattern:** `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")`; `model.to(device)` (in place); `x = x.to(device)` (returns a new tensor); constants as buffers.
- **Autocast:** forward and loss inside `torch.autocast(device.type, dtype)`; `backward()` and `step()` outside; `bfloat16` on L4, `float16` with `GradScaler`, `amp_dtype=None` on the CI path.
- **Training memory (Adam, fp32 master):** $16P+(\text{bytes per activation})\cdot Be$; bf16 activations use 2 bytes, fp32 use 4.
- **Checkpoint:** model state, optimizer state, step, `torch.get_rng_state()` (+ CUDA states); certify the resume with `torch.equal` on the CPU.
- **Variance of a mean:** $\operatorname{Var}(\bar X)=\sigma^2/n$ for independent $X_i$; paired: $\operatorname{Var}(A-B)=\operatorname{Var}A+\operatorname{Var}B-2\operatorname{Cov}(A,B)$.
- **Percentile bootstrap:** `idx = torch.randint(0, n, (n_boot, n), generator=g)`; resample labels and predictions together; `torch.quantile` at $\alpha/2$, $1-\alpha/2$.
- **Seed statistics:** mean and sample standard deviation (divisor $r-1$).
- **Pseudo-labels:** confidence $\ge\tau$; per-class cap by confidence, ties to lower index; argmax ties to lower class; indices sorted.
- **Noisy student:** student trains on clean pseudo-labels with input noise $x+0.3\varepsilon$; re-initialized each round.
- **Cluster-then-label:** frozen `conv1` features; k-means from labelled class means, 20 Lloyd iterations; majority naming with lowest-index ties; empty-label clusters take the nearest labelled-class centroid's label.
- **Forward field:** $f_i=q/(\lVert s-p_i\rVert^2+h^2)$, $h=0.25$; 3% multiplicative noise.
- **Tikhonov:** $u_\lambda=(A^\top A+\lambda I)^{-1}A^\top y$, unique for $\lambda>0$; solve, never invert.
- **Physics residual:** $\lVert\mathcal F(\hat s,\hat q)-y\rVert/\lVert y\rVert$, compared with the noise floor (about $0.03$).
- **Mixture:** $f(x)=\sum_k w_k\exp(-(x-c_k)^2/(2s_k^2))$; canonical order by center; error $\min_\pi$ mean absolute difference over 9 entries.
- **Least squares:** Adam on $(w,c,\log s)$, learning rate $0.02$; baseline 500 steps from fixed init; refinement 100 steps from the regressor.

## Concept retrieval table

| Concept | Recall prompt | Redo |
|---|---|---|
| gpu-colab-l4-workflow | Move a batch; set autocast; estimate memory; resume exactly; plan a session. | p01, p02, p06, p07, p16, p23 |
| open-ended-experiment-design | State a hypothesis, baseline, ablation, budget, and decision rule. | p03, p12, p22, p24 |
| open-ended-model-evaluation | Bootstrap a metric; pair a comparison; audit a report. | p08, p15, p21, p24, p28 |
| semi-supervised-pseudo-labeling | Select pseudo-labels; run a noisy student; name clusters; diagnose a trace. | p04, p09, p17, p18, p25 |
| scientific-ml-inverse-problems | Build $\mathcal F$ and $A$; solve Tikhonov; train and check a learned inverse. | p10, p13, p19, p26 |
| mixture-parameter-regression | Canonicalize; use a permutation-invariant metric; predict then refine. | p05, p11, p14, p20, p27 |

## Self-quiz

1. Why does `x.to(device)` on its own line not move `x`?
2. A traceback names `mat1` in `addmm` with devices `cuda:0` and `cpu`. Which tensor was not moved?
3. Under CPU `bfloat16` autocast, what dtypes do the logits, the loss, and the weight gradients have?
4. $P=10^6$, $e=10^4$, $B=100$: compute the ratio of bf16-activation to all-fp32 training memory.
5. Name the four parts of a complete checkpoint.
6. Why must a resumed run restore the global RNG state when batches come from `torch.randint`?
7. Write a falsifiable hypothesis for adding input-noise augmentation to the inverse MLP.
8. Three seeds give $0.010, 0.012, 0.014$. Give the mean and the sample standard deviation.
9. Why must labels be resampled together with predictions in a bootstrap?
10. State $\operatorname{Var}(\bar X)$ for $n$ independent values with variance $\sigma^2$, and say which Book 1 facts give it.
11. With $\tau=0.9$ and a cap of 1, which of indices 0 ($0.95$) and 2 ($0.95$), both predicted class 1, is kept?
12. Why does a noisy student beat a plain student on this unit's shapes?
13. Why is $A^\top A$ singular for the $8\times81$ grid operator, and what makes $A^\top A+\lambda I$ invertible?
14. What is the noise floor of the physics residual, and why?
15. Why does plain MSE against randomly ordered mixture targets collapse the predicted centers?

## Common pitfalls, Exam connections, and forward boundary

Common pitfalls: unassigned tensor `.to`, plain-attribute constants, autocast around `backward()`, CPU `bfloat16` timing, checkpoints without RNG state, confounded ablations, unequal budgets, single seeds, population standard deviations, test-set selection, preprocessing fitted on all rows, unpaired bootstraps, pseudo-labels without caps, students identical to their teachers, inverting a singular matrix, residuals judged without a floor, sorting mixture components by weight, and uncounted refinement steps.
Exam connections: Round 2's open-ended section rewards a working pipeline, an honest baseline comparison, a locked-test score with its uncertainty, and a writeup that explains the choices; the challenge practices p26 and p27 rehearse exactly this under a step budget.
Forward-only topics (real latent-diffusion pipelines, smarter hyperparameter search, BCa intervals and conformal prediction, FixMatch and label propagation, Bayesian inversion and physics-informed networks, set prediction and EM) are not required or permitted shortcuts here.

## Collected answers

1. Tensor `.to` returns a new tensor (or the same one if already there) and never modifies the original; the result must be assigned.
2. The input batch (or an activation created inside `forward`) feeding the linear layer is on the CPU.
3. Logits `bfloat16`, loss `float32`, gradients `float32` (master weights stay `float32`).
4. $16P=16$ MB; activations $2\cdot100\cdot10^4=2$ MB versus $4$ MB; ratio $18/20=9/10$.
5. Model `state_dict`, optimizer `state_dict`, step counter, RNG state(s).
6. Otherwise the generator is elsewhere and the resumed run samples different batches, so it is a different experiment and cannot match `torch.equal`.
7. For example: "With the same 8→64→64→3 MLP, 300 steps, and seeds 0–2, adding Gaussian noise of standard deviation 0.03 to the log magnitudes during training lowers the mean validation position error by at least 5%."
8. Mean $0.012$; deviations $\pm0.002, 0$; sum of squares $8\times10^{-6}$; divide by 2: $4\times10^{-6}$; standard deviation $0.002$.
9. Each prediction must stay paired with its own label; shuffling them separately destroys the per-example agreement the metric measures.
10. $\sigma^2/n$, from `book1:variance-of-sums` with zero covariances by `book1:independence`, and $\operatorname{Var}(aY)=a^2\operatorname{Var}(Y)$.
11. Index 0: equal confidences are broken by the lower index.
12. Input noise stops the student from fitting the teacher's exact decision on each image, so it learns a smoother, noise-robust function that generalizes better than the teacher; the noise-only ablation shows the pool is needed too.
13. $\operatorname{rank}(A^\top A)=\operatorname{rank}(A)\le8<81$; for $\lambda>0$, $v^\top(A^\top A+\lambda I)v=\lVert Av\rVert^2+\lambda\lVert v\rVert^2>0$ for $v\ne0$, so it is invertible.
14. About $0.03$: the true source's readings differ from the observations by the 3% multiplicative noise, so no prediction can explain the data much better on average.
15. The same input maps to several orderings of the target, and MSE is minimized by their average, which puts every predicted center near the mean of the true centers.